# 🚀 BÁO CÁO TOÀN DIỆN: HỆ THỐNG PHÁT HIỆN MÂU THUẪN VĂN BẢN
---
Tài liệu này ghi lại quá trình thử nghiệm, đánh giá các mô hình NLI (Natural Language Inference) và các kỹ thuật truy xuất thông tin để phát hiện sai lệch dữ liệu trong các văn bản dài.


## 🛠️ PHẦN I: THIẾT LẬP HỆ THỐNG & CẤU HÌNH MÔI TRƯỜNG
Trong phần này, chúng ta tập trung vào việc cài đặt các thư viện cần thiết và cấu hình các đường dẫn mô hình, đặc biệt là việc điều chỉnh cấu trúc thư mục trên Kaggle để AutoModel có thể tải được các kiến trúc mới như ModernBERT.


In [ ]:
from huggingface_hub import login
from itertools import cycle
from llama_cpp import Llama
from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from transformers import AutoTokenizer, AutoConfig, AutoModelForSequenceClassification
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig
from typing import List, Dict, Tuple
import argparse, os, json, pickle
import faiss
import json
import logging
import numpy as np
import os
import re
import requests
import shutil
import sys
import torch
import torch.nn.functional as F
import warnings


### 2.1 Baseline: DeBERTa-Small Long NLI
Sử dụng mô hình DeBERTa để xử lý các đoạn văn bản dài và chấm điểm mâu thuẫn dựa trên logic NLI tiêu chuẩn.


In [ ]:
#!/usr/bin/env python3
"""
Document Contradiction Detection with DeBERTa-Small Long NLI
Compare two long documents and find contradictory statements
"""


class DocumentContradictionDetector:
    def __init__(self, model_name=working_model_path1):
        """
        Initialize with DeBERTa-Small Long NLI model for document comparison
        """
        print("Loading DeBERTa-Small Long NLI model...")
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        # Load DeBERTa-Small for NLI
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.model.to(self.device)
        self.model.eval()
        
        print(f"Model loaded on device: {self.device}")
    
    def split_into_sentences(self, text: str) -> List[str]:
        """Simple sentence splitting"""
        sentences = re.split(r'[.!?]+', text)
        sentences = [s.strip() for s in sentences if len(s.strip()) > 15]
        return sentences
    
    def get_contradiction_score(self, text1: str, text2: str) -> Tuple[float, float, float]:
        """
        Get NLI score between two texts
        Returns (contradiction, neutral, entailment) probabilities
        """
        inputs = self.tokenizer(
            text1,
            text2,
            truncation=True,
            padding=True,
            max_length=512,
            return_tensors="pt"
        )
        
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = self.model(**inputs)
            probs = F.softmax(outputs.logits, dim=-1)
            
            # Return (contradiction, neutral, entailment) scores
            return tuple(probs[0].cpu().numpy())
    
    def find_contradictions_semantic_matching(self, doc1: str, doc2: str,
                                            contradiction_threshold: float = 0.5) -> List[Dict]:
        """
        Smart approach: First find semantically similar sentences, then check for contradictions
        """
        sentences1 = self.split_into_sentences(doc1)
        sentences2 = self.split_into_sentences(doc2)
        
        print(f"Smart matching: {len(sentences1)} vs {len(sentences2)} sentences...")
        
        contradictions = []
        
        # For each sentence in doc1, find potentially related sentences in doc2
        for i, sent1 in enumerate(sentences1):
            # Look for sentences in doc2 that might be talking about similar topics
            words1 = set(sent1.lower().split())
            
            candidates = []
            for j, sent2 in enumerate(sentences2):
                words2 = set(sent2.lower().split())
                # Calculate simple word overlap
                overlap = len(words1.intersection(words2))
                if overlap >= 2:  # At least 2 common words
                    candidates.append((j, sent2))
            
            # Check NLI for promising candidates
            for j, sent2 in candidates:
                contradiction, neutral, entailment = self.get_contradiction_score(sent1, sent2)
                
                if contradiction > contradiction_threshold:
                    contradictions.append({
                        'doc1_sentence': sent1,
                        'doc2_sentence': sent2,
                        'doc1_index': i,
                        'doc2_index': j,
                        'contradiction_score': float(contradiction),
                        'neutral_score': float(neutral),
                        'entailment_score': float(entailment),
                        'word_overlap': len(words1.intersection(set(sent2.lower().split())))
                    })
        
        # Sort by contradiction score
        contradictions.sort(key=lambda x: x['contradiction_score'], reverse=True)
        
        return contradictions
    
  

In [ ]:


MODEL_DIR = working_model_path1

# Tải tokenizer và model
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)


### 3.1 Hệ thống Retrieval với FAISS
Chia nhỏ văn bản thành các 'spans' và sử dụng FAISS để tìm kiếm các đoạn văn bản có liên quan nhất tới giả thuyết cần kiểm tra.


In [ ]:

def split_spans(text, max_sent=6):
    sents = sent_tokenize(text)
    spans = [" ".join(sents[i:i+max_sent]) for i in range(0, len(sents), max_sent)]
    return spans

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--input", required=True, help="Input corpus JSONL, each line: {id, text}")
    parser.add_argument("--outdir", default="./index_modernbert", help="Output dir")
    parser.add_argument("--embed_model", default="nomic-ai/modernbert-embed-base", help="ModernBERT embed model")
    args = parser.parse_args()

    os.makedirs(args.outdir, exist_ok=True)

    model = SentenceTransformer(args.embed_model)
    spans, id_map = [], []
    with open(args.input, "r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)
            doc_id, text = obj["id"], obj["text"]
            for span in split_spans(text):
                spans.append(span)
                id_map.append(doc_id)

    print(f"Encoding {len(spans)} spans ...")
    emb = model.encode(spans, batch_size=32, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)

    dim = emb.shape[1]
    index = faiss.IndexFlatIP(dim)  # cosine similarity since normalized
    index.add(emb)

    faiss.write_index(index, os.path.join(args.outdir, "index.faiss"))
    with open(os.path.join(args.outdir, "spans.pkl"), "wb") as f:
        pickle.dump(spans, f)
    with open(os.path.join(args.outdir, "id_map.pkl"), "wb") as f:
        pickle.dump(id_map, f)

    print("Index built and saved to", args.outdir)

if __name__ == "__main__":
    main()


### 4.1 Suy luận chuyên sâu với ModernBERT
Áp dụng mô hình ModernBERT (State-of-the-art) để thực hiện suy luận ngữ nghĩa với độ chính xác cao hơn trên các đoạn văn đã được trích xuất.


In [ ]:

# CONFIG
nli_model_name = working_model_path1
embed_model_name = "nomic-ai/modernbert-embed-base"
max_span_sentences = 5
top_k_retrieval = 5
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load models
tokenizer = AutoTokenizer.from_pretrained(nli_model_name, local_files_only=True)
nli_model = AutoModelForSequenceClassification.from_pretrained(nli_model_name, local_files_only=True).to(device)
embed_model = SentenceTransformer(embed_model_name)

def split_spans(text, max_sentences=5):
    sents = sent_tokenize(text)
    spans = []
    for i in range(0, len(sents), max_sentences):
        span = " ".join(sents[i:i+max_sentences])
        spans.append(span)
    return spans

def score_span(span, hypothesis):
    # cross-encoder scoring entailment/neutral/contradiction
    enc = tokenizer(span, hypothesis, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
    with torch.no_grad():
        out = nli_model(**enc)
        probs = F.softmax(out.logits, dim=-1).cpu().numpy()[0]
    # probs order assume [entailment, neutral, contradiction]; tùy model mapping
    return probs

def retrieve_evidence(premise, hypothesis):
    # split into spans
    spans = split_spans(premise, max_span_sentences)
    # encoding spans + hypothesis via embedding model to retrieve top K
    span_embs = embed_model.encode(spans, normalize_embeddings=True)
    hyp_emb = embed_model.encode([hypothesis], normalize_embeddings=True)[0]
    # cosine similarity
    sims = (span_embs @ hyp_emb)  # nếu emb được normalize => dot là cosine
    # chọn top_k spans
    top_idx = np.argsort(sims)[-top_k_retrieval:][::-1]
    retrieved = [(spans[i], sims[i]) for i in top_idx]
    return retrieved

def predict_entailment(premise, hypothesis):
    # evidence retrieval
    retrieved_spans = retrieve_evidence(premise, hypothesis)
    # cross-encoder rerank
    span_scores = []
    for span, sim in retrieved_spans:
        probs = score_span(span, hypothesis)
        span_scores.append((span, probs))
    # aggregate
    # vd: chọn span có prob entailment cao nhất
    best = max(span_scores, key=lambda x: x[1][0])  # index 0 = entailment
    return {
        "best_span": best[0],
        "entail_prob": float(best[1][0]),
        "neutral_prob": float(best[1][1]),
        "contra_prob": float(best[1][2])
    }

# === Ví dụ dùng ===
premise = """(đoạn dài như bạn gửi)"""
hypothesis = "She died on 4 January 1989 by being set on fire."  # ví dụ
res = predict_entailment(premise, hypothesis)
print("Best span:", res["best_span"])
print("Entailment: {:.3f}, Neutral: {:.3f}, Contradiction: {:.3f}".format(
    res["entail_prob"], res["neutral_prob"], res["contra_prob"]
))


### KẾT QUẢ CHẠY DEMO
Phần này hiển thị kết quả đối soát thực tế giữa các tài liệu thử nghiệm.


In [14]:

class DocumentContradictionDetector:
    def __init__(self, model_name=working_model_path1, max_words_per_span=40):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.model.to(self.device)
        self.model.eval()
        self.max_words_per_span = max_words_per_span

    def split_into_spans(self, text: str):
        raw_sentences = re.split(r'[.!?]+', text)
        raw_sentences = [s.strip() for s in raw_sentences if len(s.strip()) > 15]
        spans = []
        for sent in raw_sentences:
            words = sent.split()
            for i in range(0, len(words), self.max_words_per_span):
                span = " ".join(words[i:i+self.max_words_per_span])
                spans.append(span)
        return spans

    def get_nli_scores(self, text1, text2):
        inputs = self.tokenizer(
            text1,
            text2,
            truncation=True,
            padding=True,
            max_length=512,
            return_tensors="pt"
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = self.model(**inputs)
            probs = F.softmax(outputs.logits, dim=-1)
            return tuple(probs[0].cpu().numpy())  # (contradiction, neutral, entailment)

    def find_contradictions(self, doc1, doc2, contradiction_threshold=0.5, multi_hop=False):
        spans1 = self.split_into_spans(doc1)
        spans2 = self.split_into_spans(doc2)
        contradictions = []

        # --- Single-hop ---
        for i, span1 in enumerate(spans1):
            words1 = set(span1.lower().split())
            for j, span2 in enumerate(spans2):
                words2 = set(span2.lower().split())
                if len(words1.intersection(words2)) >= 2:
                    c, n, e = self.get_nli_scores(span1, span2)
                    if c > contradiction_threshold:
                        contradictions.append({
                            'doc1_span': span1,
                            'doc2_span': span2,
                            'doc1_index': i,
                            'doc2_index': j,
                            'contradiction_score': float(c),
                            'neutral_score': float(n),
                            'entailment_score': float(e),
                            'multi_hop': False
                        })

        # --- Multi-hop ---
        if multi_hop:
            bigrams1 = [spans1[k] + " " + spans1[k+1] for k in range(len(spans1)-1)]
            bigrams2 = [spans2[k] + " " + spans2[k+1] for k in range(len(spans2)-1)]
            for i, span1 in enumerate(bigrams1):
                for j, span2 in enumerate(bigrams2):
                    c, n, e = self.get_nli_scores(span1, span2)
                    if c > contradiction_threshold:
                        contradictions.append({
                            'doc1_span': span1,
                            'doc2_span': span2,
                            'doc1_index': f"{i}-{i+1}",
                            'doc2_index': f"{j}-{j+1}",
                            'contradiction_score': float(c),
                            'neutral_score': float(n),
                            'entailment_score': float(e),
                            'multi_hop': True
                        })

        contradictions.sort(key=lambda x: x['contradiction_score'], reverse=True)
        return contradictions

    def generate_report(self, doc1, doc2, max_contradictions=5, multi_hop=True):
        contradictions = self.find_contradictions(doc1, doc2, multi_hop=multi_hop)
        contradictions = contradictions[:max_contradictions]

        report = f"=== CONTRADICTION ANALYSIS REPORT ===\n"
        report += f"Contradictions found: {len(contradictions)}\n\n"
        for i, c in enumerate(contradictions, 1):
            tag = "MULTI-HOP" if c['multi_hop'] else "SINGLE-HOP"
            report += f"#{i} [{tag}] (Score: {c['contradiction_score']:.3f})\n"
            report += f"Doc1: {c['doc1_span']}\n"
            report += f"Doc2: {c['doc2_span']}\n"
            report += f"Entailment: {c['entailment_score']:.3f} | Neutral: {c['neutral_score']:.3f}\n"
            report += "-"*60 + "\n"
        return report


def demo():
    detector = DocumentContradictionDetector()
    doc1 = """
    Interrogator: Mr. Adams, multiple investors testified that they received returns funded directly by newer participants.
    Suspect: I thought that was common practice in high-yield funds. I didn't know it was illegal.
    Interrogator: So you're acknowledging the returns were not from actual profits?
    Suspect: Yes, the fund wasn’t generating revenue through real investments. I wanted to keep it afloat.
    """

    doc2 = """
    Interrogator: Mr. Harris, the records show your firm didn’t achieve consistent profits from operations.
    Suspect: That's correct. We had ongoing challenges with generating stable revenue.
    Interrogator: Yet, you continued to distribute returns to investors during that time?
    Suspect: Yes, we used our remaining funds and short-term loans to maintain payouts.
    Interrogator: So the payouts were not based on real business success?
    Suspect: No, they weren’t. We hoped we’d recover and make up for it eventually.
    """

    print(detector.generate_report(doc1, doc2, max_contradictions=5, multi_hop=True))


if __name__ == "__main__":
    demo()


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/598M [00:00<?, ?B/s]

W0926 18:52:03.794000 36 torch/_inductor/utils.py:1137] [1/0] Not enough SMs to use max_autotune_gemm mode


=== CONTRADICTION ANALYSIS REPORT ===
Contradictions found: 3

#1 [SINGLE-HOP] (Score: 0.991)
Doc1: Interrogator: Mr
Doc2: Interrogator: Mr
Entailment: 0.000 | Neutral: 0.009
------------------------------------------------------------
#2 [SINGLE-HOP] (Score: 0.919)
Doc1: Interrogator: So you're acknowledging the returns were not from actual profits
Doc2: Interrogator: So the payouts were not based on real business success
Entailment: 0.008 | Neutral: 0.072
------------------------------------------------------------
#3 [MULTI-HOP] (Score: 0.653)
Doc1: Interrogator: So you're acknowledging the returns were not from actual profits Suspect: Yes, the fund wasn’t generating revenue through real investments
Doc2: Interrogator: So the payouts were not based on real business success Suspect: No, they weren’t
Entailment: 0.063 | Neutral: 0.284
------------------------------------------------------------



### 4.1 Suy luận chuyên sâu với ModernBERT
Áp dụng mô hình ModernBERT (State-of-the-art) để thực hiện suy luận ngữ nghĩa với độ chính xác cao hơn trên các đoạn văn đã được trích xuất.


In [ ]:


# Model cross-encoder ModernBERT
model_name = working_model_path1
tokenizer = AutoTokenizer.from_pretrained(model_name, local_files_only=True)
model = AutoModelForSequenceClassification.from_pretrained(model_name, local_files_only=True)
model.eval()

# Hàm chạy NLI 1 cặp (premise, hypothesis)
def run_nli(premise, hypothesis):
    inputs = tokenizer(premise, hypothesis, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)[0].cpu().numpy()
    return {
        "entailment": float(probs[0]),
        "neutral": float(probs[1]),
        "contradiction": float(probs[2]),
    }

# Hàm split văn bản thành spans ngắn
def split_into_spans(text, max_len=50):
    words = text.split()
    spans = []
    for i in range(0, len(words), max_len):
        spans.append(" ".join(words[i:i+max_len]))
    return spans

# Multi-hop concat: nối nhiều spans lại
def multi_hop_concat(spans, hypothesis):
    concat_premise = " ".join(spans)
    return run_nli(concat_premise, hypothesis)

# Multi-hop chain: xét từng cặp spans
def multi_hop_chain(spans, hypothesis):
    results = []
    for i in range(len(spans)):
        for j in range(i+1, len(spans)):
            pair_premise = spans[i] + " " + spans[j]
            results.append(((i, j), run_nli(pair_premise, hypothesis)))
    return results

# =============================
# Ví dụ chạy thử
# =============================
premise = """
On 4 January 1989, after losing money in a game of mahjong the night before, 
Miyano decided to take his anger out on Furuta. He ignited a candle and dripped hot wax on her face, 
placed two shortened candles on her eyelids, and forced her to drink her own urine. 
Furuta was lifted and kicked, fell onto a stereo unit, and began a fit of convulsions. 
To prevent them from being stained with blood, the group covered their hands in plastic bags 
before beating her with their fists and an iron exercise ball, and dropped the ball on her abdomen several times. 
Miyano poured lighter fluid on Furuta and set her on fire; she made weak attempts to put herself out, 
but soon stopped moving. The assault lasted for about two hours, after which Furuta died at 10 a.m.
"""

hypothesis = "Furuta was tortured until she died."

# B1: Chia premise thành spans
spans = split_into_spans(premise, max_len=40)
print("Số spans:", len(spans))

# B2: Tính NLI cho từng span riêng lẻ
rerank_results = []
for s in spans:
    rerank_results.append({"span": s, "probs": run_nli(s, hypothesis)})

# B3: Multi-hop concat
concat_probs = multi_hop_concat(spans, hypothesis)

# B4: Multi-hop chain (cặp spans)
chain_results = multi_hop_chain(spans, hypothesis)

# =============================
# Tổng hợp kết quả
# =============================
entail_key = "entailment"
best_span = max(rerank_results, key=lambda r: r["probs"][entail_key])
mean_entail = np.mean([r["probs"][entail_key] for r in rerank_results])

best_chain = None
if chain_results:  # có ít nhất 1 cặp
    best_chain = max(chain_results, key=lambda r: r[1][entail_key])

print("\n=== Kết quả ===")
print("Best single span entail:", best_span["probs"][entail_key])
if best_chain:
    print("Best pair chain entail:", best_chain[1][entail_key], " -- spans:", best_chain[0])
else:
    print("Best pair chain entail: N/A (not enough spans)")
print("Concat entail:", concat_probs[entail_key])
print("Mean entail (single spans):", mean_entail)

print("\nBest single span text:\n", best_span["span"])
if best_chain:
    print("\nBest chain spans text:\n", spans[best_chain[0][0]], "\n---\n", spans[best_chain[0][1]])


### 5.1 Phân tích đa tầng (Multi-hop Reasoning)
Kỹ thuật kết hợp thông tin từ nhiều đoạn văn khác nhau để tìm ra các mâu thuẫn không hiển thị rõ ràng trong một câu đơn lẻ.


In [7]:

model_path = "/kaggle/input/model5"

# -----------------------------
# 1. Load tokenizer, config, model
# -----------------------------
tokenizer = AutoTokenizer.from_pretrained(
    model_path, 
    tokenizer_class="ModernBertTokenizer",  # Ép kiểu ModernBERT
    local_files_only=True
)

config = AutoConfig.from_pretrained(
    model_path, 
    trust_remote_code=True,
    local_files_only=True
)

model = AutoModelForSequenceClassification.from_pretrained(
    model_path, 
    config=config, 
    trust_remote_code=True,
    local_files_only=True
)
model.eval()

# -----------------------------
# 2. Hàm chạy NLI cho 1 cặp (premise, hypothesis)
# -----------------------------
def run_nli(premise, hypothesis):
    inputs = tokenizer(
        premise, hypothesis, 
        return_tensors="pt", truncation=True, max_length=512, padding=True
    )
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)[0].cpu().numpy()
    return {
        "entailment": float(probs[0]),
        "neutral": float(probs[1]),
        "contradiction": float(probs[2]),
    }

# -----------------------------
# 3. Chia văn bản thành spans
# -----------------------------
def split_into_spans(text, max_len=50):
    words = text.split()
    spans = []
    for i in range(0, len(words), max_len):
        spans.append(" ".join(words[i:i+max_len]))
    return spans

# -----------------------------
# 4. Multi-hop concat: gộp tất cả spans làm premise
# -----------------------------
def multi_hop_concat(spans, hypothesis):
    concat_premise = " ".join(spans)
    return run_nli(concat_premise, hypothesis)

# -----------------------------
# 5. Multi-hop chain: xét từng cặp spans
# -----------------------------
def multi_hop_chain(spans, hypothesis):
    results = []
    for i in range(len(spans)):
        for j in range(i+1, len(spans)):
            pair_premise = spans[i] + " " + spans[j]
            results.append(((i, j), run_nli(pair_premise, hypothesis)))
    return results

# -----------------------------
# 6. Ví dụ chạy thử
# -----------------------------
premise = """
On 4 January 1989, after losing money in a game of mahjong the night before, 
Miyano decided to take his anger out on Furuta. He ignited a candle and dripped hot wax on her face, 
placed two shortened candles on her eyelids, and forced her to drink her own urine. 
Furuta was lifted and kicked, fell onto a stereo unit, and began a fit of convulsions. 
To prevent them from being stained with blood, the group covered their hands in plastic bags 
before beating her with their fists and an iron exercise ball, and dropped the ball on her abdomen several times. 
Miyano poured lighter fluid on Furuta and set her on fire; she made weak attempts to put herself out, 
but soon stopped moving. The assault lasted for about two hours, after which Furuta died at 10 a.m.
"""

hypothesis = "Furuta was tortured until she died."

# Chia premise thành spans
spans = split_into_spans(premise, max_len=40)
print("Số spans:", len(spans))

# Tính NLI cho từng span riêng lẻ
rerank_results = []
for s in spans:
    rerank_results.append({"span": s, "probs": run_nli(s, hypothesis)})

# Multi-hop concat
concat_probs = multi_hop_concat(spans, hypothesis)

# Multi-hop chain (cặp spans)
chain_results = multi_hop_chain(spans, hypothesis)

# -----------------------------
# 7. Tổng hợp kết quả
# -----------------------------
entail_key = "entailment"
best_span = max(rerank_results, key=lambda r: r["probs"][entail_key])
mean_entail = np.mean([r["probs"][entail_key] for r in rerank_results])

best_chain = None
if chain_results:
    best_chain = max(chain_results, key=lambda r: r[1][entail_key])

print("\n=== Kết quả ===")
print("Best single span entail:", best_span["probs"][entail_key])
if best_chain:
    print("Best pair chain entail:", best_chain[1][entail_key], " -- spans:", best_chain[0])
else:
    print("Best pair chain entail: N/A (not enough spans)")
print("Concat entail:", concat_probs[entail_key])
print("Mean entail (single spans):", mean_entail)

print("\nBest single span text:\n", best_span["span"])
if best_chain:
    print("\nBest chain spans text:\n", spans[best_chain[0][0]], "\n---\n", spans[best_chain[0][1]])


ValueError: Unrecognized model in /kaggle/input/model5. Should have a `model_type` key in its config.json, or contain one of the following strings in its name: albert, align, altclip, aria, aria_text, audio-spectrogram-transformer, autoformer, aya_vision, bamba, bark, bart, beit, bert, bert-generation, big_bird, bigbird_pegasus, biogpt, bit, bitnet, blenderbot, blenderbot-small, blip, blip-2, blip_2_qformer, bloom, bridgetower, bros, camembert, canine, chameleon, chinese_clip, chinese_clip_vision_model, clap, clip, clip_text_model, clip_vision_model, clipseg, clvp, code_llama, codegen, cohere, cohere2, colpali, conditional_detr, convbert, convnext, convnextv2, cpmant, csm, ctrl, cvt, d_fine, dab-detr, dac, data2vec-audio, data2vec-text, data2vec-vision, dbrx, deberta, deberta-v2, decision_transformer, deepseek_v3, deformable_detr, deit, depth_anything, depth_pro, deta, detr, diffllama, dinat, dinov2, dinov2_with_registers, distilbert, donut-swin, dpr, dpt, efficientformer, efficientnet

### 1.1 Quản lý dữ liệu trên Kaggle
Thực hiện sao chép mô hình từ `/kaggle/input` vào `/kaggle/working` để có quyền chỉnh sửa file `config.json`.


In [22]:

# -----------------------------
# Ẩn warnings và traceback
# -----------------------------
warnings.filterwarnings("ignore")
sys.tracebacklimit = 0
logging.disable(logging.WARNING)

# -----------------------------
# 1. Copy model về thư mục working và sửa config ModernBERT
# -----------------------------
src_model_path = "/kaggle/input/model5"
working_model_path = "/kaggle/working/model5"
working_model_path1 = "/kaggle/working/model5/transformers/default/1"

shutil.copytree(src_model_path, working_model_path, dirs_exist_ok=True)

config_file = os.path.join(working_model_path1, "config.json")
with open(config_file, "r") as f:
    cfg = json.load(f)
cfg["model_type"] = "modernbert"
with open(config_file, "w") as f:
    json.dump(cfg, f)

# -----------------------------
# 2. Load ModernBERT NLI model
# -----------------------------
tokenizer = AutoTokenizer.from_pretrained(working_model_path1, local_files_only=True, local_files_only=True)
config = AutoConfig.from_pretrained(working_model_path1, local_files_only=True, local_files_only=True)
model = AutoModelForSequenceClassification.from_pretrained(
    working_model_path1, config=config, local_files_only=True
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# -----------------------------
# 3. Load Falcon-H1-3B-Instruct via llama_cpp
# -----------------------------
llm = Llama.from_pretrained(
    repo_id="tiiuae/Falcon-H1-3B-Instruct-GGUF",
    filename="Falcon-H1-3B-Instruct-BF16.gguf",
)

# -----------------------------
# 4. Hàm tạo summary từ Falcon-H1-3B
# -----------------------------
def generate_hypothesis_summary(text):
    prompt = (
        "Summarize all the actions the suspect has performed in this block based on the question-answer conversation. "
        "Focus on main actions, ignore emotions and unnecessary words.\n\n"
        f"{text}\n\nSummary of actions:"
    )
    out = llm(prompt, max_tokens=100, stop=["\n"])
    summary = out["choices"][0]["text"].strip()
    return summary

# -----------------------------
# 5. Hàm chạy NLI cho cặp (premise, hypothesis)
# -----------------------------
def run_nli(premise, hypothesis):
    inputs = tokenizer(
        premise, hypothesis, return_tensors="pt", truncation=True, max_length=512, padding=True
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)[0].cpu().numpy()
    return {
        "entailment": float(probs[0]),
        "neutral": float(probs[1]),
        "contradiction": float(probs[2]),
    }

# -----------------------------
# 6. Chia premise thành spans theo từ
# -----------------------------
def split_into_spans(text, max_len=50):
    words = text.split()
    spans = [" ".join(words[i:i+max_len]) for i in range(0, len(words), max_len)]
    return spans

# -----------------------------
# 7. Chia hypothesis theo block Interrogator -> Suspect
# -----------------------------
def split_qa_blocks(text):
    lines = text.splitlines()
    blocks = []
    current_block = ""
    for line in lines:
        line = line.strip()
        if line.startswith("Interrogator:"):
            if current_block:
                blocks.append(current_block.strip())
            current_block = line
        elif line.startswith("Suspect:"):
            current_block += " " + line
    if current_block:
        blocks.append(current_block.strip())
    return blocks

# -----------------------------
# 8. Multi-hop concat và chain
# -----------------------------
def multi_hop_concat(spans, hypothesis):
    concat_premise = " ".join(spans)
    return run_nli(concat_premise, hypothesis)

def multi_hop_chain(spans, hypothesis):
    results = []
    for i in range(len(spans)):
        for j in range(i+1, len(spans)):
            pair_premise = spans[i] + " " + spans[j]
            results.append(((i, j), run_nli(pair_premise, hypothesis)))
    return results

# -----------------------------
# 9. Ví dụ dữ liệu
# -----------------------------
premise = """On 4 January 1989, after losing money in a game of mahjong the night before, 
Miyano decided to take his anger out on Furuta. He ignited a candle and dripped hot wax on her face, 
placed two shortened candles on her eyelids, and forced her to drink her own urine. 
Furuta was lifted and kicked, fell onto a stereo unit, and began a fit of convulsions. 
To prevent them from being stained with blood, the group covered their hands in plastic bags 
before beating her with their fists and an iron exercise ball, and dropped the ball on her abdomen several times. 
Miyano poured lighter fluid on Furuta and set her on fire; she made weak attempts to put herself out, 
but soon stopped moving. The assault lasted for about two hours, after which Furuta died at 10 a.m."""

hypothesis = """Interrogator: On the night of January 4th, what were you doing after the mahjong game?
Suspect: I was angry about losing, but I don’t remember doing anything extreme.
Interrogator: Several witnesses indicate that you poured hot wax on Furuta’s face and placed candles on her eyelids. Can you explain why you would do that?
Suspect: I… I don’t recall that.
Interrogator: After that, it is reported that she was forced to drink her own urine, kicked, and fell onto a stereo unit, leading to convulsions. Why would you inflict such acts on her?
Suspect: I… I don’t know…
Interrogator: There are reports that you and others covered your hands in plastic bags and beat her with fists and an iron exercise ball, then set her on fire using lighter fluid. She died as a result. Can you tell me what happened during those two hours?
Suspect: I… I can’t remember…
Interrogator: Do you understand that your actions directly caused her death at 10 a.m.?"""

# -----------------------------
# 10. Chia premise & hypothesis thành spans/blocks
# -----------------------------
premise_spans = split_into_spans(premise, max_len=40)
hypothesis_blocks = split_qa_blocks(hypothesis)

# Tạo summary cho từng block
hypothesis_summaries = []
for block in hypothesis_blocks:
    try:
        summary = generate_hypothesis_summary(block)
        hypothesis_summaries.append({"original": block, "summary": summary})
    except RuntimeError:
        hypothesis_summaries.append({"original": block, "summary": ""})

# -----------------------------
# 11. Tính NLI
# -----------------------------
rerank_results = []
for s, hyp in zip(premise_spans, hypothesis_summaries):
    probs = run_nli(s, hyp["summary"])
    rerank_results.append({
        "premise_span": s,
        "hypothesis_summary": hyp["summary"],
        "hypothesis_original": hyp["original"],
        "probs": probs
    })

concat_probs = multi_hop_concat(premise_spans, " ".join([h["summary"] for h in hypothesis_summaries]))
chain_results = multi_hop_chain(premise_spans, " ".join([h["summary"] for h in hypothesis_summaries]))

# -----------------------------
# 12. Tổng hợp kết quả
# -----------------------------
entail_key = "entailment"
neutral_key = "neutral"
contrad_key = "contradiction"

best_span = max(rerank_results, key=lambda r: r["probs"][entail_key])
mean_entail = np.mean([r["probs"][entail_key] for r in rerank_results])

best_chain = None
if chain_results:
    best_chain = max(chain_results, key=lambda r: r[1][entail_key])

# -----------------------------
# 13. In kết quả
# -----------------------------
print("=== Single spans comparison ===")
for idx, r in enumerate(rerank_results, 1):
    print(f"# {idx}")
    print("Premise span:\n", r["premise_span"])
    print("Hypothesis original:\n", r["hypothesis_original"])
    print("Hypothesis summary:\n", r["hypothesis_summary"])
    print("Probs -> Entailment: {:.3f} | Neutral: {:.3f} | Contradiction: {:.3f}".format(
        r["probs"][entail_key], r["probs"][neutral_key], r["probs"][contrad_key]))
    print("-"*60)

print("\n=== Multi-hop concat ===")
print("Premise (all spans concatenated):\n", " ".join(premise_spans))
print("Hypothesis summary (all concatenated):\n", " ".join([h["summary"] for h in hypothesis_summaries]))
print("Probs -> Entailment: {:.3f} | Neutral: {:.3f} | Contradiction: {:.3f}".format(
    concat_probs[entail_key], concat_probs[neutral_key], concat_probs[contrad_key]))

if best_chain:
    i, j = best_chain[0]
    probs = best_chain[1]
    print("\n=== Best multi-hop chain ===")
    print("Spans indices:", i, j)
    print("Span texts:\n", premise_spans[i], "\n---\n", premise_spans[j])
    print("Probs -> Entailment: {:.3f} | Neutral: {:.3f} | Contradiction: {:.3f}".format(
        probs[entail_key], probs[neutral_key], probs[contrad_key]))

print("\n=== Best single span (highest entailment) ===")
print("Premise span:\n", best_span["premise_span"])
print("Hypothesis original:\n", best_span["hypothesis_original"])
print("Hypothesis summary:\n", best_span["hypothesis_summary"])
print("Probs -> Entailment: {:.3f} | Neutral: {:.3f} | Contradiction: {:.3f}".format(
    best_span["probs"][entail_key], best_span["probs"][neutral_key], best_span["probs"][contrad_key]))

print("\nMean entail (single spans):", mean_entail)


llama_model_loader: loaded meta data with 41 key-value pairs and 547 tensors from /root/.cache/huggingface/hub/models--tiiuae--Falcon-H1-3B-Instruct-GGUF/snapshots/18cc9812739f6040f795ddf9d92c9da9a8551572/./Falcon-H1-3B-Instruct-BF16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = falcon-h1
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Falcon H1 3B Instruct
llama_model_loader: - kv   3:                           general.finetune str              = Instruct
llama_model_loader: - kv   4:                           general.basename str              = Falcon-H1
llama_model_loader: - kv   5:                         general.size_label str              = 3B
llama_model_loader: - kv

=== Single spans comparison ===
# 1
Premise span:
 On 4 January 1989, after losing money in a game of mahjong the night before, Miyano decided to take his anger out on Furuta. He ignited a candle and dripped hot wax on her face, placed two shortened candles on
Hypothesis original:
 Interrogator: On the night of January 4th, what were you doing after the mahjong game? Suspect: I was angry about losing, but I don’t remember doing anything extreme.
Hypothesis summary:
 Played mahjong, angry about losing. I don't remember anything extreme.
Probs -> Entailment: 0.000 | Neutral: 0.005 | Contradiction: 0.995
------------------------------------------------------------
# 2
Premise span:
 her eyelids, and forced her to drink her own urine. Furuta was lifted and kicked, fell onto a stereo unit, and began a fit of convulsions. To prevent them from being stained with blood, the group covered their hands in
Hypothesis original:
 Interrogator: Several witnesses indicate that you poured hot wax on Fu

### 1.1 Quản lý dữ liệu trên Kaggle
Thực hiện sao chép mô hình từ `/kaggle/input` vào `/kaggle/working` để có quyền chỉnh sửa file `config.json`.


In [2]:


src_model_path = "/kaggle/input/model5"
working_model_path = "/kaggle/working/model5"
working_model_path1 = "/kaggle/working/model5/transformers/default/1"

shutil.copytree(src_model_path, working_model_path, dirs_exist_ok=True)

config_file = os.path.join(working_model_path1, "config.json")
with open(config_file, "r") as f:
    cfg = json.load(f)
cfg["model_type"] = "modernbert"
with open(config_file, "w") as f:
    json.dump(cfg, f)

-
tokenizer = AutoTokenizer.from_pretrained(working_model_path1, local_files_only=True, local_files_only=True)
config = AutoConfig.from_pretrained(working_model_path1, local_files_only=True, local_files_only=True)
model = AutoModelForSequenceClassification.from_pretrained(working_model_path1, config=config, local_files_only=True, local_files_only=True)
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Label mapping:", config.id2label)


def run_nli(premise, hypothesis):
    inputs = tokenizer(premise, hypothesis, return_tensors="pt", truncation=True, max_length=512, padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)[0].cpu().numpy()
    labels = {v: k for k, v in config.id2label.items()}
    return {
        "entailment": float(probs[labels["entailment"]]),
        "neutral": float(probs[labels["neutral"]]),
        "contradiction": float(probs[labels["contradiction"]])
    }


def split_interrogator_suspect(text):
    pattern = re.compile(r"(Interrogator:.*?)(?=Interrogator:|$)", re.DOTALL)
    blocks = pattern.findall(text)
    pairs = []
    for block in blocks:
        lines = block.strip().split("\n")
        inter, sus = None, None
        for line in lines:
            line = line.strip()
            if line.startswith("Interrogator:"):
                if inter is not None and sus is not None:
                    pairs.append((inter, sus))
                    inter, sus = line, None
                else:
                    inter = line
            elif line.startswith("Suspect:"):
                sus = line
        if inter is not None and sus is not None:
            pairs.append((inter, sus))
    return pairs

# -----------------------------
# 5. Phát hiện mâu thuẫn (bỏ threshold)
# -----------------------------
def find_contradictions(doc1, doc2):
    pairs1 = split_interrogator_suspect(doc1)
    pairs2 = split_interrogator_suspect(doc2)
    results = []

    for q1, a1 in pairs1:
        for q2, a2 in pairs2:
            probs = run_nli(q1 + " " + a1, q2 + " " + a2)
            results.append({
                "doc1_pair": (q1, a1),
                "doc2_pair": (q2, a2),
                "entailment_score": probs["entailment"],
                "neutral_score": probs["neutral"],
                "contradiction_score": probs["contradiction"]
            })
    return results

# -----------------------------
# 6. In báo cáo chi tiết
# -----------------------------
def generate_report(doc1, doc2):
    comparisons = find_contradictions(doc1, doc2)
    report = "=== CONTRADICTION REPORT ===\n"
    report += f"Total comparisons: {len(comparisons)}\n\n"
    for idx, c in enumerate(comparisons, 1):
        report += f"#{idx}\n"
        report += f"Doc1 Question: {c['doc1_pair'][0]}\n"
        report += f"Doc1 Answer: {c['doc1_pair'][1]}\n"
        report += f"Doc2 Question: {c['doc2_pair'][0]}\n"
        report += f"Doc2 Answer: {c['doc2_pair'][1]}\n"
        report += f"Scores -> Entailment: {c['entailment_score']:.3f}, Neutral: {c['neutral_score']:.3f}, Contradiction: {c['contradiction_score']:.3f}\n"
        report += "-"*80 + "\n"
    return report
def generate_contradiction_highlight_report(doc1, doc2, threshold=0.6):
    comparisons = find_contradictions(doc1, doc2)
    report = f"=== CONTRADICTION HIGHLIGHT REPORT (contradiction > {threshold}) ===\n\n"
    count = 0

    for idx, c in enumerate(comparisons, 1):
        if c["contradiction_score"] > threshold:
            count += 1
            report += f"#{count}\n"
            report += f"Doc1 Question: {c['doc1_pair'][0]}\n"
            report += f"Doc1 Answer: 🔴 {c['doc1_pair'][1]}\n"
            report += f"Doc2 Question: {c['doc2_pair'][0]}\n"
            report += f"Doc2 Answer: 🔴 {c['doc2_pair'][1]}\n"
            report += f"Scores -> Entailment: {c['entailment_score']:.3f}, Neutral: {c['neutral_score']:.3f}, Contradiction: {c['contradiction_score']:.3f}\n"
            report += "-"*80 + "\n"

    if count == 0:
        report += "⚠️ No strong contradictions found.\n"
    return report
# -----------------------------
# 7. Demo
# -----------------------------
def demo():
    doc1 = """
    Interrogator: Mr. Adams, multiple investors testified that they received returns funded directly by newer participants.
    Suspect: I thought that was common practice in high-yield funds. I didn't know it was illegal.
    Interrogator: So you're acknowledging the returns were not from actual profits?
    Suspect: Yes, the fund wasn’t generating revenue through real investments. I wanted to keep it afloat.
    """
    doc2 = """
    Interrogator: Mr. Harris, the records show your firm didn’t achieve consistent profits from operations.
    Suspect: That's correct. We had ongoing challenges with generating stable revenue.
    Interrogator: Yet, you continued to distribute returns to investors during that time?
    Suspect: Yes, we used our remaining funds and short-term loans to maintain payouts.
    Interrogator: So the payouts were not based on real business success?
    Suspect: No, they weren’t. We hoped we’d recover and make up for it eventually.
    """
    print(generate_report(doc1, doc2))

if __name__ == "__main__":
    demo()


SyntaxError: invalid syntax (2664475501.py, line 22)

In [3]:
!pip install llama-cpp-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 MB 37.3 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.1 MB/s eta 0:00:00
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.3.16-cp311-cp311-linux_x86_64.whl size=4503263 sha256=a36581ee38eb16104820ea433d1a24729953429a14892b030c34124378f5996c
  Stored in directory: /root/.cache/pip/wheels/d8/5b/e5/a7d4b5765da347d314e8155197440c9995a962f8e4a5f52b23
Successfully built llama-cpp-python


In [1]:

llm = Llama(model_path="/root/.cache/huggingface/hub/models--tiiuae--Falcon-H1-3B-Instruct-GGUF/snapshots/18cc9812739f6040f795ddf9d92c9da9a8551572/Falcon-H1-3B-Instruct-BF16.gguf")

output = llm(
    "Hello, how are you?",
    max_tokens=50,
    temperature=0.7,
    stop=["\n"]
)

print(output['choices'][0]['text'])


ModuleNotFoundError: No module named 'llama_cpp'

In [ ]:
def extract_contradictory_segments(self, doc1: str, doc2: str, 
                                   max_contradictions: int = 10) -> Dict:
    """
    Main function to extract contradictory segments from two documents
    """
    contradictions = self.find_contradictions_semantic_matching(doc1, doc2)

    # Limit results
    contradictions = contradictions[:max_contradictions]

    # Create summary
    if contradictions:
        avg_contradiction_score = np.mean([c['contradiction_score'] for c in contradictions])
        max_contradiction_score = max([c['contradiction_score'] for c in contradictions])
    else:
        avg_contradiction_score = 0
        max_contradiction_score = 0

    return {
        'contradictions': contradictions,
        'total_found': len(contradictions),
        'avg_contradiction_score': float(avg_contradiction_score),
        'max_contradiction_score': float(max_contradiction_score),
        'doc1_sentences': len(self.split_into_sentences(doc1)),
        'doc2_sentences': len(self.split_into_sentences(doc2))
    }

def generate_contradiction_report(self, doc1: str, doc2: str, 
                                   max_contradictions: int = 5) -> str:
    """
    Generate a readable report of contradictions found
    """
    result = self.extract_contradictory_segments(doc1, doc2, max_contradictions)

    report = f"""=== CONTRADICTION ANALYSIS REPORT ===

Documents analyzed:
Document 1: {result['doc1_sentences']} sentences
Document 2: {result['doc2_sentences']} sentences

Contradictions found: {result['total_found']}
Average contradiction score: {result['avg_contradiction_score']:.3f}
Highest contradiction score: {result['max_contradiction_score']:.3f}
"""
    if result['contradictions']:
        report += "=== TOP CONTRADICTIONS ===\n\n"
        
        for i, contradiction in enumerate(result['contradictions'][:max_contradictions], 1):
            report += f"CONTRADICTION #{i} (Score: {contradiction['contradiction_score']:.3f})\n"
            report += f"Doc1: {contradiction['doc1_sentence']}\n"
            report += f"Doc2: {contradiction['doc2_sentence']}\n"
            report += f"Entailment: {contradiction['entailment_score']:.3f} | Neutral: {contradiction['neutral_score']:.3f}\n"
            report += "-" * 80 + "\n\n"
    else:
        report += "No significant contradictions found.\n"

    return report


### KẾT QUẢ CHẠY DEMO
Phần này hiển thị kết quả đối soát thực tế giữa các tài liệu thử nghiệm.


In [ ]:
def demo():
    detector = DocumentContradictionDetector()
    # Two documents about the same topic but with contradictory information
    document1 = """
    Interrogator: Mr. Adams, multiple investors testified that they received returns funded directly by newer participants.
    Suspect: I thought that was common practice in high-yield funds. I didn't know it was illegal.
    Interrogator: So you're acknowledging the returns were not from actual profits?
    Suspect: Yes, the fund wasn’t generating revenue through real investments. I wanted to keep it afloat.
    Interrogator: That aligns with evidence we've gathered. Victims were misled about the source of their profits.
    Suspect: I never meant to defraud anyone, but I see now that’s exactly what happened.
    """

    document2 = """
    Interrogator: Mr. Harris, the records show your firm didn’t achieve consistent profits from operations.
    Suspect: That's correct. We had ongoing challenges with generating stable revenue.
    Interrogator: Yet, you continued to distribute returns to investors during that time?
    Suspect: Yes, we used our remaining funds and short-term loans to maintain payouts.
    Interrogator: So the payouts were not based on real business success?
    Suspect: No, they weren’t. We hoped we’d recover and make up for it eventually.
    """

    print("=== DOCUMENT CONTRADICTION DETECTION DEMO ===\n")

    # Generate report
    report = detector.generate_contradiction_report(document1, document2, max_contradictions=5)
    print(report)

    # Show detailed analysis
    print("\n=== DETAILED ANALYSIS ===")
    result = detector.extract_contradictory_segments(document1, document2, max_contradictions=3)

    for i, contradiction in enumerate(result['contradictions'][:3], 1):
        print(f"\nContradiction {i}:")
        print(f"Score: {contradiction['contradiction_score']:.3f}")
        print(f"Document 1: '{contradiction['doc1_sentence'][:100]}...'")
        print(f"Document 2: '{contradiction['doc2_sentence'][:100]}...'")

if __name__ == "__main__":
    demo()


In [ ]:
dùng FFAIS

In [ ]:
# === Ví dụ dùng ===
premise = """On the evening of 25 November 1988, Miyano and Minato rode around Misato on their motorcycles with the intention of robbing and raping local women, and spotted Furuta, who was on her way home from her part-time job. Acting on Miyano's orders, Minato kicked Furuta off her bicycle and fled the scene. Miyano, under the pretense of witnessing the attack by coincidence, approached Furuta and offered to walk her home. After further gaining her trust, Miyano walked Furuta to a nearby warehouse and threatened her, telling her that he was a yakuza member and that he would spare her only if she followed his orders.[5][7]

That night, Miyano took Furuta by taxi to a hotel in Adachi, where he raped her. He later called Minato's house and bragged to Ogura about the rape, after which Ogura told him not to let Furuta leave. In the early morning hours of 26 November, Miyano took Furuta to a park near the hotel, where Ogura, Minato, and Watanabe were waiting. They told her they knew where she lived, and that the yakuza would kill her family if she attempted to escape. Minato agreed to allow Furuta to be confined in a room on the second floor of his house in Adachi for the purpose of gang raping her. Furuta was held captive a total of 40 days.[5][7]

On 27 November, Furuta's parents contacted the police about her disappearance. To discourage further investigation, the kidnappers forced Furuta to call her mother three times to convince her that she had run away but was safe and staying with friends. When Minato's parents were present at the house where she was being confined, Furuta was forced to act as his girlfriend.[9] The group dropped this pretense when it became clear that Minato's parents would not report them to the police. The parents later claimed that they did not intervene because they were afraid of their son, who had been increasingly violent toward them.[10]

On the night of 28 November, Miyano and the others, along with Nakamura and Ihara, gang raped Furuta, after which Miyano shaved her pubic hair with a razor and used a match to burn her genital area. In early December, as punishment for an escape attempt, the group repeatedly punched Furuta in the face, and Miyano burned her ankles with a lighter. They forced Furuta to dance to music while naked, masturbate in front of them, and stand on the balcony in the middle of the night with little clothing, and inserted objects into her vagina and anus, including a metal rod and a bottle. They also forced her to drink large amounts of alcohol, milk, and water, smoke two cigarettes at once, and inhale paint thinner fumes. In one attack in the middle of the month, Furuta was beaten by the group on the pretext that Miyano had stepped on a puddle of her urine, after which he burned her thighs and hands several times with lighter fluid. From around this time, Furuta, unable to bear the repeated assaults, would sometimes plead to be killed by her captors.[5][7]

Throughout the rest of December, the severity of Furuta's abuse continued to escalate, and by the end of the month she was severely malnourished after being fed only small amounts of food and eventually only milk. Due to her injuries, she had become unable to walk to the downstairs toilet, and was confined to the room's floor in a state of extreme weakness. Her appearance had been disfigured by the beatings, with her face becoming swollen to the point of unrecognizability, and her wounds had started to emit a foul odor.[5][7]

On 4 January 1989, after losing money in a game of mahjong the night before, Miyano decided to take his anger out on Furuta. He ignited a candle and dripped hot wax on her face, placed two shortened candles on her eyelids, and forced her to drink her own urine. Furuta was lifted and kicked, fell onto a stereo unit, and began a fit of convulsions. To prevent them from being stained with blood, the group covered their hands in plastic bags before beating her with their fists and an iron exercise ball, and dropped the ball on her abdomen several times. Miyano poured lighter fluid on Furuta and set her on fire; she made weak attempts to put herself out, but soon stopped moving. The assault lasted for about two hours, after which Furuta died at 10 a.m.[5][7]

Less than 24 hours after her death, Minato's brother called to tell him that Furuta appeared to be dead. Afraid that their crime would be discovered, the group wrapped Furuta's body in a blanket and placed it in a large travel bag, then put the bag in a metal drum and filled it with wet concrete. At around 8:00 p.m. on 5 January, the group drove to a vacant lot near a construction site on the island of Wakasu in Kōtō, Tokyo, and dumped the drum there.[5][7]

In early 1989, Miyano and Ogura were arrested for kidnapping and gang raping the 19-year-old woman in December 1988. When police interrogated Miyano, he wrongly believed that Ogura had already confessed to Furuta's murder and that the police were aware of this, so he told them where to find her body. The police were initially puzzled by his confession, as they were questioning him about a different gang-rape. The drum containing Furuta's body was recovered on 29 March, and she was identified via fingerprints. Minato, Watanabe, Minato's brother, Nakamura, and Ihara were also arrested.[5][7] với 1 đoạn dài như này thfi sao """
hypothesis = "Furuta died on 4 January 1989 ."  # ví dụ
res = predict_entailment(premise, hypothesis)
print("Best span:", res["best_span"])
print("Entailment: {:.3f}, Neutral: {:.3f}, Contradiction: {:.3f}".format(
    res["entail_prob"], res["neutral_prob"], res["contra_prob"]
))
print(hypothesis)


In [11]:
print(os.listdir("/kaggle/input"))


['model2.0', 'bertforcrime']


In [2]:
login(new_session=False)

### 5.1 Phân tích đa tầng (Multi-hop Reasoning)
Kỹ thuật kết hợp thông tin từ nhiều đoạn văn khác nhau để tìm ra các mâu thuẫn không hiển thị rõ ràng trong một câu đơn lẻ.


In [3]:

def run_nli(premise, hypothesis):
    inputs = tokenizer(premise, hypothesis, return_tensors="pt", truncation=True, max_length=512, padding=True)
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)[0].cpu().numpy()
    return {
        "entailment": float(probs[0]),
        "neutral": float(probs[1]),
        "contradiction": float(probs[2]),
    }

# -----------------------------
# 4. Chia văn bản thành spans
# -----------------------------
def split_into_spans(text, max_len=50):
    words = text.split()
    spans = []
    for i in range(0, len(words), max_len):
        spans.append(" ".join(words[i:i+max_len]))
    return spans

# -----------------------------
# 5. Multi-hop concat: gộp tất cả spans làm premise
# -----------------------------
def multi_hop_concat(spans, hypothesis):
    concat_premise = " ".join(spans)
    return run_nli(concat_premise, hypothesis)

# -----------------------------
# 6. Multi-hop chain: xét từng cặp spans
# -----------------------------
def multi_hop_chain(spans, hypothesis):
    results = []
    for i in range(len(spans)):
        for j in range(i+1, len(spans)):
            pair_premise = spans[i] + " " + spans[j]
            results.append(((i, j), run_nli(pair_premise, hypothesis)))
    return results

# -----------------------------
# 7. Ví dụ chạy thử
# -----------------------------
premise = """
On 4 January 1989, after losing money in a game of mahjong the night before, 
Miyano decided to take his anger out on Furuta. He ignited a candle and dripped hot wax on her face, 
placed two shortened candles on her eyelids, and forced her to drink her own urine. 
Furuta was lifted and kicked, fell onto a stereo unit, and began a fit of convulsions. 
To prevent them from being stained with blood, the group covered their hands in plastic bags 
before beating her with their fists and an iron exercise ball, and dropped the ball on her abdomen several times. 
Miyano poured lighter fluid on Furuta and set her on fire; she made weak attempts to put herself out, 
but soon stopped moving. The assault lasted for about two hours, after which Furuta died at 10 a.m.
"""

hypothesis = """Furuta was tortured until she died."""

# Chia premise thành spans
spans = split_into_spans(premise, max_len=40)
print("Số spans:", len(spans))

# Tính NLI cho từng span riêng lẻ
rerank_results = []
for s in spans:
    rerank_results.append({"span": s, "probs": run_nli(s, hypothesis)})

# Multi-hop concat
concat_probs = multi_hop_concat(spans, hypothesis)

# Multi-hop chain (cặp spans)
chain_results = multi_hop_chain(spans, hypothesis)

# -----------------------------
# 8. Tổng hợp kết quả
# -----------------------------
entail_key = "entailment"
best_span = max(rerank_results, key=lambda r: r["probs"][entail_key])
mean_entail = np.mean([r["probs"][entail_key] for r in rerank_results])

best_chain = None
if chain_results:
    best_chain = max(chain_results, key=lambda r: r[1][entail_key])

print("\n=== Kết quả ===")
print("Best single span entail:", best_span["probs"][entail_key])
if best_chain:
    print("Best pair chain entail:", best_chain[1][entail_key], " -- spans:", best_chain[0])
else:
    print("Best pair chain entail: N/A (not enough spans)")
print("Concat entail:", concat_probs[entail_key])
print("Mean entail (single spans):", mean_entail)

print("\nBest single span text:\n", best_span["span"])
if best_chain:
    print("\nBest chain spans text:\n", spans[best_chain[0][0]], "\n---\n", spans[best_chain[0][1]])


2025-09-27 00:15:49.784779: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1758932150.165806      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758932150.259572      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Số spans: 4

=== Kết quả ===
Best single span entail: 0.9469957947731018
Best pair chain entail: 0.9999854564666748  -- spans: (1, 3)
Concat entail: 0.9999693632125854
Mean entail (single spans): 0.2527297168271616

Best single span text:
 out, but soon stopped moving. The assault lasted for about two hours, after which Furuta died at 10 a.m.

Best chain spans text:
 her eyelids, and forced her to drink her own urine. Furuta was lifted and kicked, fell onto a stereo unit, and began a fit of convulsions. To prevent them from being stained with blood, the group covered their hands in 
---
 out, but soon stopped moving. The assault lasted for about two hours, after which Furuta died at 10 a.m.


In [13]:
premise = """
On 4 January 1989, after losing money in a game of mahjong the night before, 
Miyano decided to take his anger out on Furuta. He ignited a candle and dripped hot wax on her face, 
placed two shortened candles on her eyelids, and forced her to drink her own urine. 
Furuta was lifted and kicked, fell onto a stereo unit, and began a fit of convulsions. 
To prevent them from being stained with blood, the group covered their hands in plastic bags 
before beating her with their fists and an iron exercise ball, and dropped the ball on her abdomen several times. 
Miyano poured lighter fluid on Furuta and set her on fire; she made weak attempts to put herself out, 
but soon stopped moving. The assault lasted for about two hours, after which Furuta died at 10 a.m.
"""

hypothesis = "Furuta was tortured until she died.  "


### KẾT QUẢ CHẠY DEMO
Phần này hiển thị kết quả đối soát thực tế giữa các tài liệu thử nghiệm.


In [28]:

# ==============================
# API keys AtlasCloudAI
# ==============================
API_KEYS = [
    "apikey-a6654ab1a72c47ed936c9e8b3751e2f3",
    "apikey-7f9faae05faf43ec8109578fc0131068",
    "apikey-78285455581e4487a85ded83fea42679",
    "apikey-051c6add91b649ef8d1b0191df2e6f00",
    "apikey-ec36919944024c5e88160a18b8e9494d",
    "apikey-0eef593c2650482a8d9c1a2e4eb1e28e",
    "apikey-eaeec8f00ee443cfb1e30080105bd5c2",
]

# ==============================
# Cấu hình model & endpoint
# ==============================
MODEL = "zai-org/GLM-4.5-Air"
ENDPOINT = "https://api.atlascloud.ai/v1/chat/completions"

# ==============================
# Hàm gọi AtlasCloudAI
# ==============================
def run_prompt(prompt, max_tokens=512):
    tried_keys = set()
    payload = {
        "model": MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens
    }

    while len(tried_keys) < len(API_KEYS):
        for key in API_KEYS:
            if key in tried_keys:
                continue
            headers = {"Authorization": f"Bearer {key}"}
            try:
                response = requests.post(ENDPOINT, headers=headers, json=payload, timeout=60)
                if response.status_code == 200:
                    data = response.json()
                    if "choices" in data and len(data["choices"]) > 0:
                        print(f"[SUCCESS] Key {key} trả kết quả.")
                        return data["choices"][0]["message"]["content"]
                    else:
                        print(f"[INFO] Kết quả không đúng định dạng: {data}")
                        return str(data)
                elif response.status_code == 429:
                    print(f"[RATE LIMIT] Key {key} hết quota, thử key khác...")
                    tried_keys.add(key)
                else:
                    print(f"[ERROR] Lỗi {response.status_code}: {response.text}")
                    tried_keys.add(key)
            except Exception as e:
                print(f"[EXCEPTION] Lỗi với key {key}: {e}")
                tried_keys.add(key)

    print("[FAIL] Tất cả key đều hết quota hoặc lỗi.")
    return None

# ==============================
# Demo
# ==============================
if __name__ == "__main__":
    input_text = """
    Trong bối cảnh kinh tế hiện nay, các doanh nghiệp vừa và nhỏ gặp rất nhiều khó khăn trong việc duy trì hoạt động. 
    Chi phí vận hành tăng, nguồn lực hạn chế và cạnh tranh ngày càng khốc liệt. 
    Một số doanh nghiệp đã bắt đầu tìm kiếm các giải pháp chuyển đổi số, tự động hóa và tối ưu hóa quy trình nhằm giảm chi phí và tăng hiệu quả.
    """

    prompt = f"Tóm tắt văn bản sau: {input_text.strip()}"
    summary = run_prompt(prompt, max_tokens=200)

    print("=== INPUT ===")
    print(input_text.strip())
    print("\n=== SUMMARY ===")
    print(summary.strip() if summary else "Không thể tóm tắt do hết quota.")
